In [1]:
# Import the pandas library for data manipulation.
import pandas as pd
# Import the requests library for making HTTP requests.
import requests
# Import the time module for introducing delays.
import time

# Load the clean mutations we saved in Phase 2
# Load the cleaned mutation data from the previous phase.
df = pd.read_csv("results/fh_mutations_clean.csv")
# Display the number of variants loaded.
print(f"Loaded {len(df)} variants")
# Display the first 5 rows of key columns for verification.
print(df[["gene","cdna_change","chrom","pos"]].head(5))

# Test Ensembl REST API connection
# Define a test URL for the Ensembl REST API to check connectivity.
test_url = "https://rest.ensembl.org/sequence/region/human/19:11113547..11113847"
# Set headers for plain text response.
headers = {"Content-Type": "text/plain"}
# Send a GET request to the Ensembl API.
resp = requests.get(test_url, headers=headers)

# Check for successful response.
if resp.status_code == 200:
    print(f"\nEnsembl connection: OK")
# Print a snippet of the retrieved sequence.
    print(f"Sample sequence (first 60bp): {resp.text[:60]}")
# Handle sequence retrieval failure.
else:
# Report the HTTP status code on failure.
    print(f"Connection failed: {resp.status_code}")

Loaded 27 variants
   gene cdna_change  chrom       pos
0  LDLR    c.408C>G     19  11105314
1  LDLR    c.155G>T     19  11100310
2  LDLR    c.457T>C     19  11105363
3  LDLR   c.1840T>G     19  11116993
4  LDLR   c.1054T>G     19  11110765

Ensembl connection: OK
Sample sequence (first 60bp): CAGAGCCCACGGCGTCTCTTCCTATGACACCGTCATCAGCAGAGACATCCAGGCCCCCGA


In [2]:
# Define a function to fetch a DNA sequence window around a genomic position.
def get_sequence(chrom, pos, window=300):
    """
    Fetch a DNA sequence window around a genomic position from Ensembl.
    Returns the sequence and the index of the variant within it.
    """
# Calculate the start coordinate for the sequence window.
    start = int(pos) - window
# Calculate the end coordinate for the sequence window.
    end   = int(pos) + window
    
# Construct the Ensembl API URL for the specific genomic region.
    url = f"https://rest.ensembl.org/sequence/region/human/{chrom}:{start}..{end}"
# Set headers for plain text response.
    headers = {"Content-Type": "text/plain"}
    
# Send the request to Ensembl.
    resp = requests.get(url, headers=headers)
    
# Check for successful response.
    if resp.status_code == 200:
# Return the fetched sequence and the variant position within it.
        return resp.text.strip(), window  # window = position of variant in sequence
# Handle sequence retrieval failure.
    else:
# Return None for both sequence and position.
        return None, None


# Define a function to introduce a point mutation into a wild-type sequence.
def introduce_mutation(wt_seq, position, ref, alt):
    """
    Introduce a point mutation into a wild-type sequence.
    position: index of the variant within the sequence (0-based)
    """
# Check if the wild-type sequence is valid.
    if not wt_seq:
# Return None if no wild-type sequence is provided.
        return None
    
# Convert the sequence string to a mutable list of characters.
    seq = list(wt_seq)
    
    # Verify the reference base matches what we expect
# Get the actual base at the mutation position in the wild-type sequence.
    actual = seq[position].upper()
    
# Check if the reference base matches, considering potential strand differences.
    if ref and actual != ref.upper():
        # Try the complement (some variants are on reverse strand)
# Define base complements.
        complements = {"A":"T","T":"A","G":"C","C":"G"}
# Get the complement of the reference base.
        comp = complements.get(ref.upper(), "?")
# If the actual base is the complement of the reference, it implies a reverse strand mutation.
        if actual == comp:
            # Reverse strand — flip the alt too
# Complement the alternative allele as well.
            alt = complements.get(alt.upper(), alt)
        # Continue anyway — coordinates may still be usable
    
    # Introduce the mutation
# If an alternative allele is provided.
    if alt:
# Introduce the mutation by replacing the base.
        seq[position] = alt.upper()
    
# Join the list of characters back into a string.
    return "".join(seq)


# Test on the first variant
# Select the first variant from the DataFrame for testing.
row = df.iloc[0]
print(f"Testing on: {row['gene']} {row['cdna_change']}")
print(f"Chromosome: {row['chrom']}, Position: {row['pos']}")

# Fetch wild-type sequence for the current variant.
wt_seq, var_pos = get_sequence(row["chrom"], row["pos"], window=300)

# Check if sequence retrieval was successful.
if wt_seq:
    print(f"\nWT sequence retrieved: {len(wt_seq)} bp")
# Display the 0-based index of the variant within the window.
    print(f"Variant position in window: {var_pos}")
# Display the base at the variant position.
    print(f"Base at variant position: {wt_seq[var_pos]}")
# Display a snippet of the wild-type sequence around the variant.
    print(f"WT window (middle 30bp): ...{wt_seq[285:315]}...")
# Handle sequence retrieval failure.
else:
# Report failure.
    print("Failed to retrieve sequence")

Testing on: LDLR c.408C>G
Chromosome: 19, Position: 11105314

WT sequence retrieved: 601 bp
Variant position in window: 300
Base at variant position: C
WT window (middle 30bp): ...CCGGGACTGCTTGGACGGCTCAGACGAGGC...


In [3]:
# Initialize a list to store results for all variants.
results = []

# Iterate through each row (variant) in the DataFrame.
for i, row in df.iterrows():
    print(f"Fetching {i+1}/27: {row['gene']} {row['cdna_change']}...")
    
    # Fetch wild-type sequence
# Fetch wild-type sequence for the current variant.
    wt_seq, var_pos = get_sequence(row["chrom"], row["pos"], window=300)
    
# Handle cases where sequence retrieval fails.
    if wt_seq is None:
# Report failure and skip to the next variant.
        print(f"  FAILED — skipping")
# Continue to the next iteration.
        continue
    
    # Parse ref and alt from cdna_change (e.g. c.408C>G → ref=C, alt=G)
# Get the cDNA change string.
    cdna = row["cdna_change"]
# Initialize reference and alternative alleles.
    ref, alt = "", ""
# Check if the cDNA change indicates a substitution (e.g., C>G).
    if ">" in cdna:
# Extract the part of the string containing the change (e.g., "408C>G").
        change_part = cdna.split(".")[-1]  # e.g. "408C>G"
# Attempt to parse ref and alt alleles.
        try:
            # Extract letters around the >
# Extract reference allele (alphabetic characters before ">").
            ref = "".join([c for c in change_part.split(">")[0] if c.isalpha()])
# Extract alternative allele (alphabetic characters after ">").
            alt = "".join([c for c in change_part.split(">")[1] if c.isalpha()])
# Handle parsing errors.
        except:
# Continue if parsing fails, ref and alt will remain empty.
            pass
    
    # Introduce the mutation
# Introduce the mutation into the wild-type sequence.
    mut_seq = introduce_mutation(wt_seq, var_pos, ref, alt)
    
    # Check if mutation actually changed anything
# Check if the mutation actually resulted in a sequence change.
    changed = (wt_seq != mut_seq) if mut_seq else False
    
# Append the results for the current variant.
    results.append({
# ClinVar ID.
        "clinvar_id":   row["clinvar_id"],
# Gene symbol.
        "gene":         row["gene"],
# cDNA change.
        "cdna_change":  row["cdna_change"],
# Protein change.
        "protein_change": row["protein_change"],
# Chromosome.
        "chrom":        row["chrom"],
# Genomic position.
        "pos":          row["pos"],
# Reference allele.
        "ref":          ref,
# Alternative allele.
        "alt":          alt,
# Wild-type sequence.
        "wt_seq":       wt_seq,
# Mutated sequence.
        "mut_seq":      mut_seq,
# Variant position in sequence.
        "var_pos":      var_pos,
# Boolean indicating if sequence was changed.
        "seq_changed":  changed,
    })
    
    # Be polite to Ensembl API — small pause between requests
# Pause to avoid overwhelming the Ensembl API with requests.
    time.sleep(0.3)

# Create a DataFrame from the collected sequence data.
df_seqs = pd.DataFrame(results)

print(f"\nSequences retrieved: {len(df_seqs)}")
print(f"Mutations introduced successfully: {df_seqs['seq_changed'].sum()}")
print(f"Failed/unchanged: {(~df_seqs['seq_changed']).sum()}")
print(f"\nSample:")
# Print the first 10 rows of key columns.
print(df_seqs[["gene","cdna_change","ref","alt","seq_changed"]].head(10))

Fetching 1/27: LDLR c.408C>G...
Fetching 2/27: LDLR c.155G>T...
Fetching 3/27: LDLR c.457T>C...
Fetching 4/27: LDLR c.1840T>G...
Fetching 5/27: LDLR c.1054T>G...
Fetching 6/27: LDLR c.1961T>G...
Fetching 7/27: LDLR c.1865A>T...
Fetching 8/27: LDLR c.1724T>A...
Fetching 9/27: LDLR c.1074T>G...
Fetching 10/27: LDLR c.986G>C...
Fetching 11/27: LDLR c.648T>G...
Fetching 12/27: LDLR c.398A>G...
Fetching 13/27: LDLR c.483C>G...
Fetching 14/27: LDLR c.1067A>G...
Fetching 15/27: LDLR NM_000527.5(LDLR):c.818-1G>T...
Fetching 16/27: LDLR c.1297G>A...
Fetching 17/27: LDLR c.325T>A...
Fetching 18/27: LDLR c.348C>A...
Fetching 19/27: LDLR c.632A>G...
Fetching 20/27: LDLR c.270T>G...
Fetching 21/27: LDLR NM_000527.5(LDLR):c.68-2A>C...
Fetching 22/27: APOB c.9721G>T...
Fetching 23/27: APOB c.11470C>T...
Fetching 24/27: APOB c.6022G>T...
Fetching 25/27: APOB c.5303C>A...
Fetching 26/27: APOB c.8602G>T...
Fetching 27/27: PCSK9 NM_174936.4(PCSK9):c.399+1G>A...

Sequences retrieved: 27
Mutations introduc

In [4]:
# Section for saving the processed sequence data.
# Save sequences to CSV
# Note: sequences are long so we save a summary version and a full version
# Save the full DataFrame, including sequences, to a CSV file.
df_seqs.to_csv("results/fh_sequences.csv", index=False)

# Save a readable summary without the long sequences
# Create a summary DataFrame by dropping the long sequence columns.
df_summary = df_seqs.drop(columns=["wt_seq", "mut_seq"])
# Save the summary DataFrame to a separate CSV file.
df_summary.to_csv("results/fh_sequences_summary.csv", index=False)

# Inform about saved files.
print("Files saved:")
# Details of the full data file.
print("  results/fh_sequences.csv         — full data including sequences")
# Details of the summary data file.
print("  results/fh_sequences_summary.csv — summary without sequences")
print()

# Section for a quick verification of mutation introduction.
# Quick sanity check — show one WT vs mutant comparison
# Select the first sample from the sequence DataFrame.
sample = df_seqs.iloc[0]
# Get the variant position for the sample.
pos = sample["var_pos"]
print(f"Sanity check — {sample['gene']} {sample['cdna_change']}:")
print(f"  WT  at position {pos}: ...{sample['wt_seq'][pos-5:pos+6]}...")
print(f"  MUT at position {pos}: ...{sample['mut_seq'][pos-5:pos+6]}...")
print(f"  Change: {sample['ref']} → {sample['alt']}")
print()
# Indicate completion of Phase 3.
print("Phase 3 complete!")

Files saved:
  results/fh_sequences.csv         — full data including sequences
  results/fh_sequences_summary.csv — summary without sequences

Sanity check — LDLR c.408C>G:
  WT  at position 300: ...TTGGACGGCTC...
  MUT at position 300: ...TTGGAGGGCTC...
  Change: C → G

Phase 3 complete!
